# Foods YOLOv8m — upload zip (no Google Drive mount)

1. On your PC use: `data/processed/object_detect/foods_yolo_dataset.zip`
2. Runtime → **T4 GPU**
3. Run all cells — when asked, **upload the zip** (~200 MB)
4. Download `best.pt` at the end → save to `ml/models/object_detect/foods/best.pt`

In [ ]:
!pip install -q ultralytics

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

print("Select foods_yolo_dataset.zip from your PC...")
uploaded = files.upload()
zip_path = Path(next(iter(uploaded)))

extract = Path("/content/foods_dataset")
if extract.exists():
    import shutil
    shutil.rmtree(extract)
extract.mkdir(parents=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract)

if (extract / "data.yaml").is_file():
    FOODS_DIR = extract
else:
    FOODS_DIR = next(p.parent for p in extract.rglob("data.yaml"))

yaml = FOODS_DIR / "data.yaml"
yaml.write_text(
    f"path: {FOODS_DIR.as_posix()}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    "nc: 3\n"
    "names: [rice, oil, canned_fish]\n",
    encoding="utf-8",
)

TRAIN_DATA = str(yaml)
print("FOODS_DIR:", FOODS_DIR)
print("Train images:", len(list((FOODS_DIR / "train/images").glob("*.jpg"))))
print(yaml.read_text())

In [ ]:
from ultralytics import YOLO

RUNS = Path("/content/runs/yolo")
RUNS.mkdir(parents=True, exist_ok=True)
NAME = "foods_yolov8m_colab"

YOLO("yolov8m.pt").train(
    data=TRAIN_DATA,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    workers=8,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    warmup_epochs=3,
    close_mosaic=10,
    project=str(RUNS),
    name=NAME,
    exist_ok=True,
    plots=True,
    val=True,
)

BEST_PT = RUNS / NAME / "weights" / "best.pt"
print("best.pt:", BEST_PT)

In [ ]:
from IPython.display import Image, display

rp = RUNS / NAME / "results.png"
if rp.is_file():
    display(Image(filename=str(rp), width=900))
print(YOLO(str(BEST_PT)).val(data=TRAIN_DATA, imgsz=640, device=0))

In [ ]:
files.download(str(BEST_PT))